In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
def get_counts_data(input_dict, sl_no):  
    counts = {'Sl. No.': sl_no, 'PROPERTY': 0, 'UL_PROPERTY': 0, 'UL_SUB_PROPERTY': 0, 'FILLER': 0, 'TOTAL_LOAD': 0, 
              'AUTO_CERT': 0, 'AUTO_SERIES': [], 'RAILWAY_CERT': 0, 'HAZARD_LEVELS': [], 'REQ_SETS': [], 
              'WATER_CERT': 0, 'TEMP': [], 'INDUSTRY': 0, 'REGION': 0}     
    
    combination = [] # Initialize empty list for combination  
    nested_fields = ['PROPERTY', 'FILLER', 'AUTO_CERT', 'RAILWAY_CERT', 'WATER_CERT']
    for key, value in input_dict.items():
        if key not in nested_fields and value:
            counts[key] = len(value) 
            combination.append(key)
        elif key == 'PROPERTY' and value:
            for prop in value:
                if prop['property_type'] == 'property':
                    counts['PROPERTY'] += 1
                elif prop['property_type'] == 'ul_property':
                    counts['UL_PROPERTY'] += 1
                elif prop['property_type'] == 'ul_sub_property':
                    counts['UL_SUB_PROPERTY'] += 1
                    
        elif key == 'FILLER' and value:
            for f in value:
                if 'filler_name' in f:
                    counts['FILLER'] += 1
                else:
                    counts['TOTAL_LOAD'] += 1
                    
        elif key == 'AUTO_CERT' and value:
            counts['AUTO_CERT'] = len(value)
            for c in value:
                counts['AUTO_SERIES'].append(len(c['CERTS']))
        elif key == 'RAILWAY_CERT' and value:
            counts['RAILWAY_CERT'] = len(value)
            for c in value:
                counts['HAZARD_LEVELS'].append(len(c['Hazard_Level']))
                counts['REQ_SETS'].append(len(c['Req_Set']))
        elif key == 'WATER_CERT' and value:
            counts['WATER_CERT'] = len(value)
            for c in value:
                counts['TEMP'].append(len(c['Temp']))
        else:
            counts[key] = 0
    
    counts['COMBINATION'] = combination
    return counts

In [3]:
import ast

def validate_property(properties): 
    for prop in properties:  
        if not isinstance(prop, dict):  
            return False, "Each item in 'PROPERTY' must be a dictionary"  
        if 'property_name' not in prop or 'modifier' not in prop or 'property_type' not in prop:  
            return False, "Each property dictionary must contain 'property_name', 'modifier', and 'property_type' keys"  
        if not isinstance(prop['modifier'], dict):  
            return False, "'modifier' must be a dictionary"  
        for key in ['value', 'min', 'max', 'unit']:  
            if key not in prop['modifier']:  
                return False, f"Missing key '{key}' in modifier"  
    return True, "" 

def validate_filler(fillers): 
    for filler in fillers:  
        if not isinstance(filler, dict):  
            return False, "Each item in 'FILLER' must be a dictionary"  
        if 'filler_name' in filler:  
            if not isinstance(filler['filler_name'], list):  
                return False, "'filler_name' must be a list"  
        elif 'total_load' in filler:  
            if not isinstance(filler['total_load'], dict):  
                return False, "'total_load' must be a dictionary"  
            for key in ['value', 'min', 'max']:  
                if key not in filler['total_load']:  
                    return False, f"Missing key '{key}' in total_load"  
        else:  
            return False, "Each filler dictionary must contain either 'filler_name' or 'total_load'" 
    return True, "" 


def validate_auto_cert(auto_certs):  
    for cert in auto_certs:  
        if not isinstance(cert, dict):  
            return False, "Each item in 'AUTO_CERT' must be a dictionary"  
        if 'OEM' not in cert or 'CERTS' not in cert:  
            return False, "Each auto cert dictionary must contain 'OEM' and 'CERTS'"  
        if not isinstance(cert['CERTS'], list):  
            return False, "'CERTS' must be a list"  
    return True, ""  
  
def validate_railway_cert(railway_certs):  
    for cert in railway_certs:  
        if not isinstance(cert, dict):  
            return False, "Each item in 'RAILWAY_CERT' must be a dictionary"  
        if 'Standard' not in cert or 'Hazard_Level' not in cert or 'Req_Set' not in cert:  
            return False, "Each railway cert dictionary must contain 'Standard', 'Hazard_Level', and 'Req_Set'"  
        if not isinstance(cert['Hazard_Level'], list) or not isinstance(cert['Req_Set'], list):  
            return False, "'Hazard_Level' and 'Req_Set' must be lists"  
    return True, ""  
  
def validate_water_cert(water_certs):  
    for cert in water_certs:  
        if not isinstance(cert, dict):  
            return False, "Each item in 'WATER_CERT' must be a dictionary"  
        if 'Standard' not in cert or 'Temp' not in cert:  
            return False, "Each water cert dictionary must contain 'Standard' and 'Temp'"  
        if not isinstance(cert['Temp'], list):  
            return False, "'Temp' must be a list"  
    return True, ""  
  
def validate_nsf_cert(nsf_certs):  
    if not isinstance(nsf_certs, list):  
        return False, "'NSF_CERT' must be a list"  
    return True, ""  
  
def validate_dict(data):   
    expected_format = {  
    'GRADE': list,  
    'APPLICATION': list,  
    'BRAND': list,  
    'POLYMER': list,  
    'PROPERTY': list,  
    'FILLER': list,  
    'FEATURE': list,  
    'PROCESSING': list,  
    'DELIVERY_FORM': list,  
    'COMPETITOR_GRADE': list,  
    'AUTO_CERT': list,  
    'RAILWAY_CERT': list,  
    'WATER_CERT': list,  
    'NSF_CERT': list,
    'INDUSTRY': list,
    'REGION': list
    }  
    
     
    # Find any entities in the input data that are not expected  
    unknown_entities = set(data.keys() - expected_format.keys())
      
    # if unknown_entities and not "CERTIFICATION" in unknown_entities:  
    if unknown_entities:  
        return False, f"Unknown entities detected: {', '.join(unknown_entities)}" 
    
#     if "CERTIFICATION" in unknown_entities and data["CERTIFICATION"]:
# # # #     if "CERTIFICATION" in unknown_entities:
#         return False, f"Value found in CERTIFICATION" 
  
    # Check if all required keys are present and of the correct type  
    for key, expected_type in expected_format.items():  
        if key not in data:  
            return False, f"Missing key: {key}"  
        if not isinstance(data[key], expected_type):  
            return False, f"Key '{key}' is not of type {expected_type.__name__}"   
    
    # Validate PROPERTY  
    valid, message = validate_property(data['PROPERTY'])  
    if not valid:  
        return False, message
    
    # Validate FILLER  
    valid, message = validate_filler(data['FILLER'])  
    if not valid:  
        return False, message

  
    # Validate AUTO_CERT  
    valid, message = validate_auto_cert(data['AUTO_CERT'])  
    if not valid:  
        return False, message  
  
    # Validate RAILWAY_CERT  
    valid, message = validate_railway_cert(data['RAILWAY_CERT'])  
    if not valid:  
        return False, message  
  
    # Validate WATER_CERT  
    valid, message = validate_water_cert(data['WATER_CERT'])  
    if not valid:  
        return False, message  
  
    # Validate NSF_CERT  
    valid, message = validate_nsf_cert(data['NSF_CERT'])  
    if not valid:  
        return False, message  
  
    return True, "Validation successful"  

In [4]:
files = {
    'file1': 'Prod_queries_from_02_19_to_03_2_reviewed',
    'file2': 'Prod_Query_1-16_to_2-18_part1_reviewed',
    'file3': 'Prod_Query_1-16_to_2-18_part2_reviewed',
    'file4': 'Prod_Query_1-16_to_2-18_part3_reviewed',
    'file5': 'labelled_prod_queries',
    'file6': 'labeled_data_for_grade_comp_grade_brand_intial_review',
    'file7': 'labeled_data_survey_queries 2_filtered',
    'file8': 'Sample labels 1 Reviewed', 
    'file9': 'enhancement_queries',
    'file10': 'prod_generated_data_27_01_25',
    'file11': 'prod_generated_data_auto_num_27_01_25',
    'file12': 'auto_certifications_data_27_01_25',
    'file13': 'multi_ul_sub_props_27_01_25',
    'file14': 'grade_cgrade_data_27_01_25',
    'file15': 'prod_generated_data_prop_grade_27_01_25',
}

df1 = pd.read_excel(f"data/reviewed/prod/{files['file1']}.xlsx")
df2 = pd.read_excel(f"data/reviewed/prod/{files['file2']}.xlsx")
df3 = pd.read_excel(f"data/reviewed/prod/{files['file3']}.xlsx")
df4 = pd.read_excel(f"data/reviewed/prod/{files['file4']}.xlsx")
df5 = pd.read_excel(f"data/reviewed/prod/{files['file5']}.xlsx")
df6 = pd.read_excel(f"data/reviewed/{files['file6']}.xlsx")
df7 = pd.read_excel(f"data/reviewed/{files['file7']}.xlsx")
df8 = pd.read_excel(f"data/reviewed/{files['file8']}.xlsx")
df9 = pd.read_excel(f"data/reviewed/{files['file9']}.xlsx")
df10 = pd.read_excel(f"data/reviewed/{files['file10']}.xlsx")
df11 = pd.read_excel(f"data/reviewed/{files['file11']}.xlsx")
df12 = pd.read_excel(f"data/reviewed/{files['file12']}.xlsx")
df13 = pd.read_excel(f"data/reviewed/{files['file13']}.xlsx")
df14 = pd.read_excel(f"data/reviewed/{files['file14']}.xlsx")
df15 = pd.read_excel(f"data/reviewed/{files['file15']}.xlsx")

In [5]:
def add_prefix(prefix_df, file_path, des_folder=None):
    name = os.path.splitext(os.path.basename(file_path))[0]
    if "Sl. No." not in list(prefix_df):
        cols = ["Sl. No."] + list(prefix_df)
        prefix_df["Sl. No."] = list(range(1, len(prefix_df)+1))
        prefix_df = prefix_df[cols]

    prefix_df['Ref No.'] = list(prefix_df['Sl. No.'].values)
    prefix_df['Ref No.'] = prefix_df['Ref No.'].apply(lambda x: f'{name}_'+str(x))
    if des_folder:
        prefix_df.to_excel(des_folder+name+'.xlsx', header=True, index=False)

    prefix_df = prefix_df[['Sl. No.', 'Query', 'Output', 'Ref No.']]
    return prefix_df

In [6]:
df1 = add_prefix(df1, files['file1'], 'data/Training_Data_27_01_25/data/prod/')  
df2 = add_prefix(df2, files['file2'], 'data/Training_Data_27_01_25/data/prod/')  
df3 = add_prefix(df3, files['file3'], 'data/Training_Data_27_01_25/data/prod/')  
df4 = add_prefix(df4, files['file4'], 'data/Training_Data_27_01_25/data/prod/')  
df5 = add_prefix(df5, files['file5'], 'data/Training_Data_27_01_25/data/prod/')  
df6 = add_prefix(df6, files['file6'], 'data/Training_Data_27_01_25/data/')  
df7 = add_prefix(df7, files['file7'], 'data/Training_Data_27_01_25/data/')  
df8 = add_prefix(df8, files['file8'], 'data/Training_Data_27_01_25/data/')  
df9 = add_prefix(df9, files['file9'], 'data/Training_Data_27_01_25/data/')  
df10 = add_prefix(df10, files['file10'], 'data/Training_Data_27_01_25/data/')  
df11 = add_prefix(df11, files['file11'], 'data/Training_Data_27_01_25/data/')  
df12 = add_prefix(df12, files['file12'], 'data/Training_Data_27_01_25/data/')  
df13 = add_prefix(df13, files['file13'], 'data/Training_Data_27_01_25/data/')  
df14 = add_prefix(df14, files['file14'], 'data/Training_Data_27_01_25/data/')
df15 = add_prefix(df15, files['file15'], 'data/Training_Data_27_01_25/data/')  
# df16 = add_prefix(df16, files['file16'], 'data/Training_Data_27_01_25/data/')   

In [7]:
# df10

In [8]:
# + len(df3) + len(df4) + len(df5) + len(df6) + len(df7) + len(df8) + len(df9) 
# len(df1) + len(df2) + len(df10) + len(df11) + len(df12) + len(df13) + len(df14) + len(df15) + len(df16) + len(df17) + len(df18) + len(df19) + len(df20) + len(df21) + len(df22)
len(df1) + len(df2) + len(df3) + len(df4) + len(df5) + len(df6) + len(df7) + len(df8) + len(df9) + len(df10) + len(df11) + len(df12) + len(df13) + len(df14) + len(df15)

62239

In [9]:
df1

,Sl. No.,Query,Output,Ref No.
0,1,tangent delta lower than 0.004,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_1
1,2,pa610 cf,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_2
2,3,f1 in ul746c,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_3
3,4,iso1304,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_4
4,5,flexible pps grades,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_5
...,...,...,...,...
262,264,pa66 with glas fibre 50%,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_264
263,265,thermal conductive,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_265
264,266,"thermal conductive,pa","{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_266
265,267,thermal shock,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_267


In [10]:
# prod_df = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8, df9, df10, df11, df12, df13, df14, df15, df16, df17, df18, df19, df20, df21, df22]).reset_index(drop=True)
# prod_df = pd.concat([df1, df2, df10, df11, df12, df13, df14, df15, df16, df17, df18, df19, df20, df21, df22]).reset_index(drop=True)
prod_df = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8, df10, df11, df12, df13, df14, df15]).reset_index(drop=True) # df9 enhancement queries added later
print(len(prod_df))
prod_df['Query'] = prod_df['Query'].str.lower()
prod_df = prod_df[~prod_df['Query'].duplicated(keep='last')]
print(len(prod_df))
prod_df = prod_df[prod_df["Query"].notna()]
print(len(prod_df))
# weightage to business queries
prod_df = pd.concat([df1, df1, df2, df2, df3, df3, df4, df4, df9, df9, prod_df]).reset_index(drop=True) # enhancement queries 2 x df9
print(len(prod_df))
cols = ["Sl. No."] + list(prod_df)
prod_df["Sl. No."] = list(range(1, len(prod_df)+1))
prod_df = prod_df[['Sl. No.', 'Query', 'Output', 'Ref No.']]
prod_df = prod_df.reset_index(drop=True)
# prod_df = prod_df[cols]
prod_df

60233
56063
56062
61532


,Sl. No.,Query,Output,Ref No.
0,1,tangent delta lower than 0.004,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_1
1,2,pa610 cf,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_2
2,3,f1 in ul746c,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_3
3,4,iso1304,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_4
4,5,flexible pps grades,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_5
...,...,...,...,...
61527,61528,what is the glass transition of fp 6e5901a80?,"{'GRADE': ['fp 6e5901a80'], 'APPLICATION': [],...",prod_generated_data_prop_grade_27_01_25_46
61528,61529,elongational stress f of lfrt cfr-tp pet gf60-10,"{'GRADE': ['lfrt cfr-tp pet gf60-10'], 'APPLIC...",prod_generated_data_prop_grade_27_01_25_47
61529,61530,tensile creep modulus at 1h of frianyl a3 v2 o...,"{'GRADE': ['frianyl a3 v2 or 2003/p'], 'APPLIC...",prod_generated_data_prop_grade_27_01_25_48
61530,61531,flexural stress at 3.5% iso 178 (mpa) of impet...,"{'GRADE': ['impet 830r'], 'APPLICATION': [], '...",prod_generated_data_prop_grade_27_01_25_49


In [11]:
df = prod_df
count_data = []
incorrect = []
# for idx, q, output in zip(df1['Sl. No.'], df1['Query'], df1["Output"]):
for idx, q, output, r in zip(df['Sl. No.'], df['Query'], df["Output"], df["Ref No."]):
    try:
        if not isinstance(output, dict):  
            output = ast.literal_eval(output)
    except Exception as e:
        print(idx, e)
        print("\n", q, r, "\n")
        print(output)
        incorrect.append(idx)
        continue
        
    is_valid, message = validate_dict(output)
    
    if not is_valid:
        print(idx, message, r)
        # print("\n", q, "\n")
        # print(output, "\n\n")
        incorrect.append(idx)
    else:
        count_data.append(get_counts_data(output, idx))
            
print(len(incorrect))
# print(incorrect)

0


In [12]:
prod_df.to_excel("data/Training_Data_27_01_25/without_formatting/Training_Data_27_01_25.xlsx", header=True, index=False)

In [13]:
prod_df

,Sl. No.,Query,Output,Ref No.
0,1,tangent delta lower than 0.004,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_1
1,2,pa610 cf,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_2
2,3,f1 in ul746c,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_3
3,4,iso1304,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_4
4,5,flexible pps grades,"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], ...",Prod_queries_from_02_19_to_03_2_reviewed_5
...,...,...,...,...
61527,61528,what is the glass transition of fp 6e5901a80?,"{'GRADE': ['fp 6e5901a80'], 'APPLICATION': [],...",prod_generated_data_prop_grade_27_01_25_46
61528,61529,elongational stress f of lfrt cfr-tp pet gf60-10,"{'GRADE': ['lfrt cfr-tp pet gf60-10'], 'APPLIC...",prod_generated_data_prop_grade_27_01_25_47
61529,61530,tensile creep modulus at 1h of frianyl a3 v2 o...,"{'GRADE': ['frianyl a3 v2 or 2003/p'], 'APPLIC...",prod_generated_data_prop_grade_27_01_25_48
61530,61531,flexural stress at 3.5% iso 178 (mpa) of impet...,"{'GRADE': ['impet 830r'], 'APPLICATION': [], '...",prod_generated_data_prop_grade_27_01_25_49


In [14]:
count_data[:5]

[{'Sl. No.': 1,
  'PROPERTY': 1,
  'UL_PROPERTY': 0,
  'UL_SUB_PROPERTY': 0,
  'FILLER': 0,
  'TOTAL_LOAD': 0,
  'AUTO_CERT': 0,
  'AUTO_SERIES': [],
  'RAILWAY_CERT': 0,
  'HAZARD_LEVELS': [],
  'REQ_SETS': [],
  'WATER_CERT': 0,
  'TEMP': [],
  'INDUSTRY': 0,
  'REGION': 0,
  'GRADE': 0,
  'APPLICATION': 0,
  'BRAND': 0,
  'POLYMER': 0,
  'FEATURE': 0,
  'PROCESSING': 0,
  'DELIVERY_FORM': 0,
  'COMPETITOR_GRADE': 0,
  'NSF_CERT': 0,
  'COMBINATION': []},
 {'Sl. No.': 2,
  'PROPERTY': 0,
  'UL_PROPERTY': 0,
  'UL_SUB_PROPERTY': 0,
  'FILLER': 1,
  'TOTAL_LOAD': 1,
  'AUTO_CERT': 0,
  'AUTO_SERIES': [],
  'RAILWAY_CERT': 0,
  'HAZARD_LEVELS': [],
  'REQ_SETS': [],
  'WATER_CERT': 0,
  'TEMP': [],
  'INDUSTRY': 0,
  'REGION': 0,
  'GRADE': 0,
  'APPLICATION': 0,
  'BRAND': 0,
  'POLYMER': 1,
  'FEATURE': 0,
  'PROCESSING': 0,
  'DELIVERY_FORM': 0,
  'COMPETITOR_GRADE': 0,
  'NSF_CERT': 0,
  'COMBINATION': ['POLYMER']},
 {'Sl. No.': 3,
  'PROPERTY': 0,
  'UL_PROPERTY': 1,
  'UL_SUB_PROP

In [15]:
idx = 5
print(df[df['Sl. No.']==idx]['Query'].values[0])
df[df['Sl. No.']==idx]['Output'].values[0]

flexible pps grades


"{'GRADE': [], 'APPLICATION': [], 'BRAND': [], 'POLYMER': ['pps'], 'PROPERTY': [], 'FILLER': [], 'FEATURE': [], 'PROCESSING': [], 'DELIVERY_FORM': [], 'COMPETITOR_GRADE': [], 'AUTO_CERT': [], 'RAILWAY_CERT': [], 'WATER_CERT': [], 'NSF_CERT': [], 'INDUSTRY': [], 'REGION': []}"

In [16]:
ignore_cols = ['AUTO_SERIES',  'COMBINATION',  'HAZARD_LEVELS', 'REQ_SETS', 'TEMP', 'TOTAL_LOAD', 'UL_SUB_PROPERTY', 'COMBINATION_2']

In [17]:
count_df = pd.DataFrame(count_data)
count_df['COMBINATION_2'] = count_df['COMBINATION'].apply(lambda x: "+".join(x)).replace("", np.NaN)
count_df

,Sl. No.,PROPERTY,UL_PROPERTY,UL_SUB_PROPERTY,FILLER,TOTAL_LOAD,AUTO_CERT,AUTO_SERIES,RAILWAY_CERT,HAZARD_LEVELS,...,APPLICATION,BRAND,POLYMER,FEATURE,PROCESSING,DELIVERY_FORM,COMPETITOR_GRADE,NSF_CERT,COMBINATION,COMBINATION_2
0,1,1,0,0,0,0,0,[],0,[],...,0,0,0,0,0,0,0,0,[],NaN
1,2,0,0,0,1,1,0,[],0,[],...,0,0,1,0,0,0,0,0,[POLYMER],POLYMER
2,3,0,1,0,0,0,0,[],0,[],...,0,0,0,0,0,0,0,0,[],NaN
3,4,0,0,0,0,0,0,[],0,[],...,0,0,0,0,0,0,0,0,[],NaN
4,5,0,0,0,0,0,0,[],0,[],...,0,0,1,0,0,0,0,0,[POLYMER],POLYMER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61527,61528,1,0,0,0,0,0,[],0,[],...,0,0,0,0,0,0,0,0,[GRADE],GRADE
61528,61529,1,0,0,0,0,0,[],0,[],...,0,0,0,0,0,0,0,0,[GRADE],GRADE
61529,61530,1,0,0,0,0,0,[],0,[],...,0,0,0,0,0,0,0,0,[GRADE],GRADE
61530,61531,1,0,0,0,0,0,[],0,[],...,0,0,0,0,0,0,0,0,[GRADE],GRADE


In [18]:
filtered_cols = [i for i in count_df.columns if i not in ignore_cols]
count_df['No_of_Entity_Tokens'] = count_df[filtered_cols].drop(['Sl. No.'], axis=1).sum(axis=1, numeric_only=True)

In [19]:
count_df['COMBINATION_2'].value_counts()

COMBINATION_2
COMPETITOR_GRADE                              20934
GRADE                                          9088
POLYMER                                        3496
BRAND                                          2428
FEATURE                                        1917
                                              ...  
APPLICATION+POLYMER+NSF_CERT                      1
APPLICATION+POLYMER+DELIVERY_FORM+INDUSTRY        1
APPLICATION+PROCESSING+NSF_CERT+INDUSTRY          1
GRADE+BRAND+POLYMER+PROCESSING                    1
FEATURE+DELIVERY_FORM                             1
Name: count, Length: 105, dtype: int64

In [20]:
list(count_df['COMBINATION_2'].value_counts().keys())

['COMPETITOR_GRADE',
 'GRADE',
 'POLYMER',
 'BRAND',
 'FEATURE',
 'POLYMER+FEATURE',
 'APPLICATION+INDUSTRY',
 'REGION',
 'BRAND+FEATURE',
 'NSF_CERT',
 'POLYMER+PROCESSING',
 'APPLICATION+POLYMER+INDUSTRY',
 'COMPETITOR_GRADE+REGION',
 'BRAND+POLYMER',
 'APPLICATION',
 'PROCESSING',
 'APPLICATION+POLYMER+FEATURE+INDUSTRY',
 'POLYMER+REGION',
 'APPLICATION+FEATURE+INDUSTRY',
 'BRAND+PROCESSING',
 'BRAND+DELIVERY_FORM',
 'APPLICATION+POLYMER+INDUSTRY+REGION',
 'APPLICATION+DELIVERY_FORM+INDUSTRY',
 'APPLICATION+BRAND+INDUSTRY',
 'POLYMER+FEATURE+REGION',
 'POLYMER+FEATURE+PROCESSING',
 'GRADE+FEATURE',
 'BRAND+FEATURE+REGION',
 'POLYMER+DELIVERY_FORM',
 'BRAND+REGION',
 'APPLICATION+POLYMER',
 'DELIVERY_FORM',
 'GRADE+POLYMER',
 'GRADE+PROCESSING',
 'BRAND+NSF_CERT',
 'GRADE+BRAND',
 'FEATURE+NSF_CERT',
 'BRAND+POLYMER+FEATURE',
 'PROCESSING+NSF_CERT',
 'FEATURE+PROCESSING',
 'BRAND+FEATURE+PROCESSING',
 'FEATURE+REGION',
 'GRADE+FEATURE+DELIVERY_FORM',
 'BRAND+POLYMER+PROCESSING',
 'AP

In [21]:
count_df.drop(['Sl. No.', 'COMBINATION', 'COMBINATION_2'], axis=1).replace(0, np.NaN).describe()

,PROPERTY,UL_PROPERTY,UL_SUB_PROPERTY,FILLER,TOTAL_LOAD,AUTO_CERT,RAILWAY_CERT,WATER_CERT,INDUSTRY,REGION,GRADE,APPLICATION,BRAND,POLYMER,FEATURE,PROCESSING,DELIVERY_FORM,COMPETITOR_GRADE,NSF_CERT,No_of_Entity_Tokens
count,6227.000000,3799.000000,2343.000000,3827.0,4525.0,4316.000000,1413.0,1321.000000,3714.000000,2597.000000,10107.000000,4509.000000,5599.000000,9256.000000,6856.000000,2173.000000,1147.000000,21611.000000,1076.000000,61127.000000
mean,1.236551,1.082390,1.045241,1.0,1.0,1.070899,1.0,1.009841,1.038234,1.014632,1.003661,1.036815,1.131452,1.076707,1.165111,1.155545,1.363557,1.000278,1.000929,1.560293
std,0.512385,0.279741,0.225607,0.0,0.0,0.284112,0.0,0.098750,0.353723,0.138008,0.060397,0.334330,0.354940,0.266951,0.389335,0.398793,0.481233,0.016660,0.030486,0.927707
min,1.000000,1.000000,1.000000,1.0,1.0,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000,1.0,1.0,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000,1.0,1.0,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
75%,1.000000,1.000000,1.000000,1.0,1.0,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,1.000000,1.000000,2.000000
max,5.000000,3.000000,4.000000,1.0,1.0,4.000000,1.0,2.000000,5.000000,3.000000,2.000000,5.000000,4.000000,3.000000,4.000000,4.000000,2.000000,2.000000,2.000000,10.000000


In [22]:
count_df.drop(['Sl. No.', 'COMBINATION', 'COMBINATION_2'], axis=1).sum(axis=0, numeric_only=True)

PROPERTY                7700
UL_PROPERTY             4112
UL_SUB_PROPERTY         2449
FILLER                  3827
TOTAL_LOAD              4525
AUTO_CERT               4622
RAILWAY_CERT            1413
WATER_CERT              1334
INDUSTRY                3856
REGION                  2635
GRADE                  10144
APPLICATION             4675
BRAND                   6335
POLYMER                 9966
FEATURE                 7988
PROCESSING              2511
DELIVERY_FORM           1564
COMPETITOR_GRADE       21617
NSF_CERT                1077
No_of_Entity_Tokens    95376
dtype: int64

In [23]:
# PROPERTY                5773
# UL_PROPERTY             3759
# UL_SUB_PROPERTY         2058
# FILLER                  3048
# TOTAL_LOAD              3538
# AUTO_CERT               3618
# RAILWAY_CERT            1263
# WATER_CERT              1098
# GRADE                   5190
# APPLICATION             4377
# BRAND                   5053
# POLYMER                 9765
# FEATURE                 8414
# PROCESSING              3236
# DELIVERY_FORM           1735
# COMPETITOR_GRADE        9474
# NSF_CERT                 986
# No_of_Entity_Tokens    66789

In [24]:
unique_combinations = []
for comb in list(count_df['COMBINATION']):
    if comb and set(comb) not in unique_combinations:
        unique_combinations.append(set(comb))
        
unique_combinations

[{'POLYMER'},
 {'FEATURE', 'POLYMER'},
 {'BRAND'},
 {'FEATURE', 'POLYMER', 'PROCESSING'},
 {'POLYMER', 'PROCESSING'},
 {'FEATURE'},
 {'APPLICATION', 'FEATURE', 'INDUSTRY', 'POLYMER'},
 {'APPLICATION', 'INDUSTRY'},
 {'APPLICATION', 'INDUSTRY', 'POLYMER'},
 {'GRADE'},
 {'FEATURE', 'GRADE'},
 {'BRAND', 'POLYMER'},
 {'APPLICATION', 'GRADE'},
 {'APPLICATION', 'DELIVERY_FORM', 'INDUSTRY'},
 {'BRAND', 'COMPETITOR_GRADE'},
 {'FEATURE', 'PROCESSING'},
 {'BRAND', 'PROCESSING'},
 {'APPLICATION', 'BRAND'},
 {'NSF_CERT', 'POLYMER'},
 {'BRAND', 'GRADE', 'POLYMER'},
 {'BRAND', 'FEATURE'},
 {'PROCESSING'},
 {'APPLICATION', 'POLYMER'},
 {'DELIVERY_FORM', 'POLYMER'},
 {'APPLICATION'},
 {'BRAND', 'FEATURE', 'PROCESSING'},
 {'NSF_CERT'},
 {'GRADE', 'POLYMER'},
 {'APPLICATION', 'FEATURE'},
 {'APPLICATION', 'FEATURE', 'POLYMER'},
 {'APPLICATION', 'BRAND', 'INDUSTRY'},
 {'COMPETITOR_GRADE'},
 {'COMPETITOR_GRADE', 'POLYMER'},
 {'DELIVERY_FORM', 'GRADE'},
 {'APPLICATION', 'BRAND', 'DELIVERY_FORM'},
 {'DELIVERY